In [1]:
from pathlib import Path

project_folder = Path(r"D:\electricity-tariff-optimisation")
raw_folder = project_folder / "data" / "raw"

if not raw_folder.exists():
    raise FileNotFoundError(f"Folder not found: {raw_folder}")

for item in sorted(raw_folder.iterdir()):
    item_type = "Folder" if item.is_dir() else "File"
    print(f"{item_type}: {item.name}")

csv_files = sorted(raw_folder.rglob("*.csv"))

print(f"\nCSV files found: {len(csv_files)}")

for file in csv_files[:5]:
    print(file.relative_to(raw_folder))

Folder: Partitioned LCL Data
File: Partitioned LCL Data.zip
File: Tariffs.xlsx

CSV files found: 168
Partitioned LCL Data\Small LCL Data\LCL-June2015v2_0.csv
Partitioned LCL Data\Small LCL Data\LCL-June2015v2_1.csv
Partitioned LCL Data\Small LCL Data\LCL-June2015v2_10.csv
Partitioned LCL Data\Small LCL Data\LCL-June2015v2_100.csv
Partitioned LCL Data\Small LCL Data\LCL-June2015v2_101.csv


In [2]:
import pandas as pd

sample_file = csv_files[0]
sample = pd.read_csv(sample_file, nrows=10)

print("Sample file:", sample_file.name)
print("\nColumns:")
print(sample.columns.tolist())

print("\nData types:")
print(sample.dtypes)

display(sample)

Sample file: LCL-June2015v2_0.csv

Columns:
['LCLid', 'stdorToU', 'DateTime', 'KWH/hh (per half hour) ']

Data types:
LCLid                      object
stdorToU                   object
DateTime                   object
KWH/hh (per half hour)      int64
dtype: object


,LCLid,stdorToU,DateTime,KWH/hh (per half hour)
0,MAC000002,Std,2012-10-12 00:30:00.0000000,0
1,MAC000002,Std,2012-10-12 01:00:00.0000000,0
2,MAC000002,Std,2012-10-12 01:30:00.0000000,0
3,MAC000002,Std,2012-10-12 02:00:00.0000000,0
4,MAC000002,Std,2012-10-12 02:30:00.0000000,0
5,MAC000002,Std,2012-10-12 03:00:00.0000000,0
6,MAC000002,Std,2012-10-12 03:30:00.0000000,0
7,MAC000002,Std,2012-10-12 04:00:00.0000000,0
8,MAC000002,Std,2012-10-12 04:30:00.0000000,0
9,MAC000002,Std,2012-10-12 05:00:00.0000000,0


In [3]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [4]:
tariff_file = raw_folder / "Tariffs.xlsx"

with pd.ExcelFile(tariff_file) as workbook:
    print("Sheet names:", workbook.sheet_names)

    for sheet_name in workbook.sheet_names:
        preview = pd.read_excel(
            workbook,
            sheet_name=sheet_name,
            nrows=10,
        )

        print(f"\nSheet: {sheet_name}")
        print("Columns:", preview.columns.tolist())
        display(preview)

Sheet names: ['Sheet1', 'Sheet2', 'Sheet3']

Sheet: Sheet1
Columns: ['TariffDateTime', 'Tariff']


,TariffDateTime,Tariff
0,2013-01-01 00:00:00,Normal
1,2013-01-01 00:30:00,Normal
2,2013-01-01 01:00:00,Normal
3,2013-01-01 01:30:00,Normal
4,2013-01-01 02:00:00,Normal
5,2013-01-01 02:30:00,Normal
6,2013-01-01 03:00:00,Normal
7,2013-01-01 03:30:00,Normal
8,2013-01-01 04:00:00,Normal
9,2013-01-01 04:30:00,Normal



Sheet: Sheet2
Columns: []


""



Sheet: Sheet3
Columns: []


""


In [5]:
tariffs = pd.read_excel(tariff_file, sheet_name=0)
tariffs.columns = tariffs.columns.str.strip()

tariffs["TariffDateTime"] = pd.to_datetime(
    tariffs["TariffDateTime"],
    errors="coerce",
)

print("Rows:", len(tariffs))
print("Start:", tariffs["TariffDateTime"].min())
print("End:", tariffs["TariffDateTime"].max())

print("\nTariff values:")
print(tariffs["Tariff"].value_counts(dropna=False))

print("\nMissing values:")
print(tariffs.isna().sum())

print(
    "\nDuplicate timestamps:",
    tariffs["TariffDateTime"].duplicated().sum(),
)

print("\nTimestamp intervals:")
print(
    tariffs["TariffDateTime"]
    .sort_values()
    .diff()
    .value_counts()
)

display(tariffs.head())

Rows: 17520
Start: 2013-01-01 00:00:00
End: 2013-12-31 23:30:00

Tariff values:
Tariff
Normal    15072
Low        1660
High        788
Name: count, dtype: int64

Missing values:
TariffDateTime    0
Tariff            0
dtype: int64

Duplicate timestamps: 0

Timestamp intervals:
TariffDateTime
0 days 00:30:00    17519
Name: count, dtype: int64


,TariffDateTime,Tariff
0,2013-01-01 00:00:00,Normal
1,2013-01-01 00:30:00,Normal
2,2013-01-01 01:00:00,Normal
3,2013-01-01 01:30:00,Normal
4,2013-01-01 02:00:00,Normal


In [6]:
first_file = pd.read_csv(
    csv_files[0],
    dtype="string",
)

first_file.columns = first_file.columns.str.strip()

first_file["timestamp"] = pd.to_datetime(
    first_file["DateTime"],
    errors="coerce",
)

first_file["energy_kwh"] = pd.to_numeric(
    first_file["KWH/hh (per half hour)"],
    errors="coerce",
)

print("File:", csv_files[0].name)
print("Rows:", len(first_file))
print("Households:", first_file["LCLid"].nunique())
print("Start:", first_file["timestamp"].min())
print("End:", first_file["timestamp"].max())

print("\nTariff groups:")
print(first_file["stdorToU"].value_counts(dropna=False))

print("\nInvalid or missing timestamps:")
print(first_file["timestamp"].isna().sum())

print("\nInvalid or missing energy readings:")
print(first_file["energy_kwh"].isna().sum())

print("\nNegative energy readings:")
print(first_file["energy_kwh"].lt(0).sum())

print("\nZero energy readings:")
print(first_file["energy_kwh"].eq(0).sum())

display(
    first_file.groupby("LCLid").agg(
        rows=("timestamp", "size"),
        start=("timestamp", "min"),
        end=("timestamp", "max"),
        valid_readings=("energy_kwh", "count"),
    ).head(10)
)

File: LCL-June2015v2_0.csv
Rows: 1000000
Households: 30
Start: 2011-12-06 13:00:00
End: 2014-02-28 00:00:00

Tariff groups:
stdorToU
Std    1000000
Name: count, dtype: Int64

Invalid or missing timestamps:
0

Invalid or missing energy readings:
29

Negative energy readings:
0

Zero energy readings:
45538


,rows,start,end,valid_readings
LCLid,,,,
MAC000002,24158,2012-10-12 00:30:00,2014-02-28,24157
MAC000003,35469,2012-02-20 13:00:00,2014-02-28,35468
MAC000004,31677,2012-05-08 13:00:00,2014-02-28,31676
MAC000006,36461,2012-01-30 11:30:00,2014-02-28,36460
MAC000007,25046,2012-09-24 12:00:00,2014-02-28,25045
MAC000008,26013,2012-05-05 09:00:00,2013-10-30,26012
MAC000009,25238,2012-09-20 10:00:00,2014-02-28,25237
MAC000010,25049,2012-09-24 12:30:00,2014-02-28,25048
MAC000011,23705,2012-10-22 11:00:00,2014-02-28,23704


In [7]:
year_data = first_file.loc[
    first_file["timestamp"].ge("2013-01-01")
    & first_file["timestamp"].lt("2014-01-01")
].copy()

year_data["valid_timestamp"] = year_data["timestamp"].where(
    year_data["energy_kwh"].notna()
)

year_data["is_zero"] = year_data["energy_kwh"].eq(0)

coverage = year_data.groupby("LCLid").agg(
    rows=("timestamp", "size"),
    unique_timestamps=("timestamp", "nunique"),
    valid_timestamps=("valid_timestamp", "nunique"),
    start=("timestamp", "min"),
    end=("timestamp", "max"),
    missing_readings=("energy_kwh", lambda values: values.isna().sum()),
    zero_readings=("is_zero", "sum"),
    total_kwh=("energy_kwh", lambda values: values.sum(min_count=1)),
)

coverage["duplicate_rows"] = (
    coverage["rows"] - coverage["unique_timestamps"]
)

coverage["coverage_pct"] = (
    coverage["valid_timestamps"] / len(tariffs) * 100
).round(2)

display(coverage.round({"total_kwh": 2}))

print("Households checked:", len(coverage))
print("Expected time slots:", len(tariffs))

,rows,unique_timestamps,valid_timestamps,start,end,missing_readings,zero_readings,total_kwh,duplicate_rows,coverage_pct
LCLid,,,,,,,,,,
MAC000002,17532,17520,17520,2013-01-01,2013-12-31 23:30:00,0.0,0,4217.56,12,100.00
MAC000003,17530,17518,17518,2013-01-01,2013-12-31 23:30:00,0.0,0,7029.26,12,99.99
MAC000004,17531,17519,17519,2013-01-01,2013-12-31 23:30:00,0.0,13451,618.75,12,99.99
MAC000006,17532,17520,17520,2013-01-01,2013-12-31 23:30:00,0.0,1156,1054.24,12,100.00
MAC000007,17529,17517,17517,2013-01-01,2013-12-31 23:30:00,0.0,0,3419.98,12,99.98
MAC000008,14505,14495,14495,2013-01-01,2013-10-30 00:00:00,0.0,3,5203.49,10,82.73
MAC000009,17523,17511,17511,2013-01-01,2013-12-31 23:30:00,0.0,0,3013.3,12,99.95
MAC000010,17532,17520,17520,2013-01-01,2013-12-31 23:30:00,0.0,0,9312.43,12,100.00
MAC000011,17531,17519,17519,2013-01-01,2013-12-31 23:30:00,0.0,0,5248.13,12,99.99


Households checked: 29
Expected time slots: 17520


In [8]:
key_columns = ["LCLid", "timestamp"]

duplicate_records = year_data.loc[
    year_data.duplicated(
        subset=key_columns,
        keep=False,
    )
].sort_values(key_columns)

duplicate_summary = (
    duplicate_records.groupby(key_columns)
    .agg(
        rows=("energy_kwh", "size"),
        distinct_values=(
            "energy_kwh",
            lambda values: values.nunique(dropna=False),
        ),
    )
    .reset_index()
)

print("Repeated household-timestamp pairs:", len(duplicate_summary))

print(
    "Pairs with conflicting readings:",
    duplicate_summary["distinct_values"].gt(1).sum(),
)

print("\nRepeated timestamps across households:")
display(
    duplicate_summary.groupby("timestamp")
    .agg(households=("LCLid", "nunique"))
    .reset_index()
)

print("\nDuplicate records for MAC000002:")
display(
    duplicate_records.loc[
        duplicate_records["LCLid"].eq("MAC000002"),
        ["LCLid", "DateTime", "energy_kwh"],
    ]
)

Repeated household-timestamp pairs: 331
Pairs with conflicting readings: 0

Repeated timestamps across households:


,timestamp,households
0,2013-01-21,28
1,2013-02-21,28
2,2013-03-24,28
3,2013-04-24,28
4,2013-05-25,28
5,2013-06-25,28
6,2013-07-26,28
7,2013-08-26,28
8,2013-09-26,28
9,2013-10-27,27



Duplicate records for MAC000002:


,LCLid,DateTime,energy_kwh
4800,MAC000002,2013-01-21 00:00:00.0000000,0.21
4801,MAC000002,2013-01-21 00:00:00.0000000,0.21
6289,MAC000002,2013-02-21 00:00:00.0000000,0.216
6290,MAC000002,2013-02-21 00:00:00.0000000,0.216
7778,MAC000002,2013-03-24 00:00:00.0000000,0.486
7779,MAC000002,2013-03-24 00:00:00.0000000,0.486
9267,MAC000002,2013-04-24 00:00:00.0000000,0.147
9268,MAC000002,2013-04-24 00:00:00.0000000,0.147
10756,MAC000002,2013-05-25 00:00:00.0000000,0.17
10757,MAC000002,2013-05-25 00:00:00.0000000,0.17


In [9]:
if duplicate_summary["distinct_values"].gt(1).any():
    raise ValueError("Conflicting duplicate readings require review.")

group_counts = (
    year_data.groupby(["LCLid", "timestamp"])["stdorToU"]
    .nunique(dropna=False)
)

if group_counts.gt(1).any():
    raise ValueError("Conflicting tariff groups require review.")

removed_mask = year_data.duplicated(
    subset=["LCLid", "timestamp"],
    keep="first",
)

removed_records = year_data.loc[removed_mask].copy()
year_clean = year_data.loc[~removed_mask].copy()

print("Rows before:", len(year_data))
print("Duplicate rows removed:", len(removed_records))
print("Rows after:", len(year_clean))

print(
    "Energy removed from duplicate rows (kWh):",
    round(removed_records["energy_kwh"].sum(), 3),
)

print(
    "Remaining duplicate keys:",
    year_clean.duplicated(["LCLid", "timestamp"]).sum(),
)

print(
    "Clean total energy (kWh):",
    round(year_clean["energy_kwh"].sum(min_count=1), 3),
)

Rows before: 484387
Duplicate rows removed: 331
Rows after: 484056
Energy removed from duplicate rows (kWh): 125.24
Remaining duplicate keys: 0
Clean total energy (kWh): 115155.077


In [10]:
expected_times = pd.DatetimeIndex(
    tariffs["TariffDateTime"]
)

quality_rows = []

for household_id, household in year_clean.groupby("LCLid"):
    observed_times = pd.DatetimeIndex(household["timestamp"])
    valid = household["energy_kwh"].notna()
    valid_count = int(valid.sum())
    zero_count = int(household["energy_kwh"].eq(0).sum())

    quality_rows.append({
        "household_id": household_id,
        "observed_slots": len(observed_times),
        "missing_slots": len(expected_times.difference(observed_times)),
        "unexpected_slots": len(observed_times.difference(expected_times)),
        "missing_readings": int((~valid).sum()),
        "zero_pct": (
            round(zero_count / valid_count * 100, 2)
            if valid_count else None
        ),
        "total_kwh": household["energy_kwh"].sum(min_count=1),
    })

quality_table = pd.DataFrame(quality_rows)

display(
    quality_table.sort_values(
        ["missing_slots", "zero_pct"],
        ascending=False,
    ).round({"total_kwh": 2})
)

print("Households checked:", len(quality_table))

,household_id,observed_slots,missing_slots,unexpected_slots,missing_readings,zero_pct,total_kwh
11,MAC000016,721,16799,0,0,0.00,22.02
22,MAC000028,13392,4128,0,0,3.84,968.41
5,MAC000008,14495,3025,0,0,0.02,5200.98
20,MAC000026,17507,13,0,0,0.00,4351.75
23,MAC000029,17508,12,0,0,2.82,1144.46
6,MAC000009,17511,9,0,0,0.00,3012.32
24,MAC000030,17511,9,0,0,0.00,4222.44
19,MAC000025,17514,6,0,0,0.27,1833.44
27,MAC000034,17516,4,0,0,0.00,8804.47
4,MAC000007,17517,3,0,0,0.00,3419.01


Households checked: 29


In [11]:
complete_ids = quality_table.loc[
    quality_table["missing_slots"].eq(0)
    & quality_table["unexpected_slots"].eq(0)
    & quality_table["missing_readings"].eq(0),
    "household_id",
].tolist()

pilot_data = year_clean.loc[
    year_clean["LCLid"].isin(complete_ids),
    ["LCLid", "stdorToU", "timestamp", "energy_kwh"],
].copy()

pilot_data = pilot_data.sort_values(
    ["LCLid", "timestamp"]
).reset_index(drop=True)

expected_rows = len(complete_ids) * len(tariffs)

assert len(pilot_data) == expected_rows
assert not pilot_data.duplicated(["LCLid", "timestamp"]).any()
assert pilot_data["energy_kwh"].notna().all()

print("Pilot households:", len(complete_ids))
print("Household IDs:", complete_ids)
print("Pilot rows:", len(pilot_data))
print("Total energy (kWh):", round(pilot_data["energy_kwh"].sum(), 3))

display(pilot_data.head())

Pilot households: 6
Household IDs: ['MAC000002', 'MAC000006', 'MAC000010', 'MAC000018', 'MAC000019', 'MAC000027']
Pilot rows: 105120
Total energy (kWh): 23785.074


,LCLid,stdorToU,timestamp,energy_kwh
0,MAC000002,Std,2013-01-01 00:00:00,0.219
1,MAC000002,Std,2013-01-01 00:30:00,0.241
2,MAC000002,Std,2013-01-01 01:00:00,0.191
3,MAC000002,Std,2013-01-01 01:30:00,0.235
4,MAC000002,Std,2013-01-01 02:00:00,0.182


In [12]:
processed_folder = project_folder / "data" / "processed"
quality_folder = project_folder / "reports" / "data_quality"

processed_folder.mkdir(parents=True, exist_ok=True)
quality_folder.mkdir(parents=True, exist_ok=True)

pilot_file = processed_folder / "pilot_consumption_2013.csv"
quality_file = quality_folder / "first_file_household_quality_2013.csv"
duplicates_file = quality_folder / "first_file_removed_duplicates_2013.csv"

pilot_data.to_csv(pilot_file, index=False)
quality_table.to_csv(quality_file, index=False)
removed_records.to_csv(duplicates_file, index=False)

print("Pilot data saved:", pilot_file)
print("Quality summary saved:", quality_file)
print("Removed duplicates saved:", duplicates_file)

Pilot data saved: D:\electricity-tariff-optimisation\data\processed\pilot_consumption_2013.csv
Quality summary saved: D:\electricity-tariff-optimisation\reports\data_quality\first_file_household_quality_2013.csv
Removed duplicates saved: D:\electricity-tariff-optimisation\reports\data_quality\first_file_removed_duplicates_2013.csv


In [13]:
from zipfile import ZipFile

zip_file = raw_folder / "Partitioned LCL Data.zip"

with ZipFile(zip_file) as archive:
    documentation_files = [
        name
        for name in archive.namelist()
        if not name.endswith("/")
        and name.lower().endswith(
            (".txt", ".md", ".pdf", ".doc", ".docx", ".xlsx")
        )
    ]

    print("Documentation files found:", len(documentation_files))

    for name in documentation_files:
        print(name)

        if name.lower().endswith((".txt", ".md")):
            content = archive.read(name).decode(
                "utf-8",
                errors="replace",
            )
            print(content[:12000])
            print()

Documentation files found: 0


In [14]:
boundary_check = first_file.loc[
    first_file["LCLid"].isin(complete_ids)
    & first_file["timestamp"].isin(
        pd.to_datetime([
            "2013-01-01 00:00:00",
            "2013-01-01 00:30:00",
            "2014-01-01 00:00:00",
        ])
    ),
    ["LCLid", "timestamp", "energy_kwh"],
].sort_values(["LCLid", "timestamp"])

display(boundary_check)

,LCLid,timestamp,energy_kwh
3840,MAC000002,2013-01-01 00:00:00,0.219
3841,MAC000002,2013-01-01 00:30:00,0.241
21372,MAC000002,2014-01-01 00:00:00,0.496
107446,MAC000006,2013-01-01 00:00:00,0.0
107447,MAC000006,2013-01-01 00:30:00,0.0
124978,MAC000006,2014-01-01 00:00:00,0.039
208792,MAC000010,2013-01-01 00:00:00,0.509
208793,MAC000010,2013-01-01 00:30:00,0.453
226324,MAC000010,2014-01-01 00:00:00,0.643
345387,MAC000018,2013-01-01 00:00:00,0.187


In [15]:
pilot_corrected = first_file.loc[
    first_file["LCLid"].isin(complete_ids)
    & first_file["timestamp"].gt("2013-01-01")
    & first_file["timestamp"].le("2014-01-01"),
    ["LCLid", "stdorToU", "timestamp", "energy_kwh"],
].copy()

keys = ["LCLid", "timestamp"]

conflicts = (
    pilot_corrected.groupby(keys)["energy_kwh"]
    .nunique(dropna=False)
)

if conflicts.gt(1).any():
    raise ValueError("Conflicting readings require review.")

pilot_corrected = (
    pilot_corrected.drop_duplicates(keys)
    .rename(columns={"timestamp": "interval_end_utc"})
    .sort_values(["LCLid", "interval_end_utc"])
    .reset_index(drop=True)
)

pilot_corrected["interval_end_utc"] = (
    pilot_corrected["interval_end_utc"].dt.tz_localize("UTC")
)

pilot_corrected["interval_start_utc"] = (
    pilot_corrected["interval_end_utc"] - pd.Timedelta(minutes=30)
)

expected_ends = pd.date_range(
    start="2013-01-01 00:30",
    end="2014-01-01 00:00",
    freq="30min",
    tz="UTC",
)

assert pilot_corrected["energy_kwh"].notna().all()
assert len(pilot_corrected) == len(complete_ids) * len(expected_ends)

for household_id, household in pilot_corrected.groupby("LCLid"):
    actual_ends = pd.DatetimeIndex(household["interval_end_utc"])
    assert actual_ends.equals(expected_ends), household_id

pilot_data = pilot_corrected
pilot_data.to_csv(pilot_file, index=False)

print("Corrected pilot saved:", pilot_file)
print("Households:", pilot_data["LCLid"].nunique())
print("Rows:", len(pilot_data))
print("Total energy (kWh):", round(pilot_data["energy_kwh"].sum(), 3))

display(pilot_data.head())

Corrected pilot saved: D:\electricity-tariff-optimisation\data\processed\pilot_consumption_2013.csv
Households: 6
Rows: 105120
Total energy (kWh): 23785.371


,LCLid,stdorToU,interval_end_utc,energy_kwh,interval_start_utc
0,MAC000002,Std,2013-01-01 00:30:00+00:00,0.241,2013-01-01 00:00:00+00:00
1,MAC000002,Std,2013-01-01 01:00:00+00:00,0.191,2013-01-01 00:30:00+00:00
2,MAC000002,Std,2013-01-01 01:30:00+00:00,0.235,2013-01-01 01:00:00+00:00
3,MAC000002,Std,2013-01-01 02:00:00+00:00,0.182,2013-01-01 01:30:00+00:00
4,MAC000002,Std,2013-01-01 02:30:00+00:00,0.229,2013-01-01 02:00:00+00:00


In [16]:
flat_rate_gbp_per_kwh = 0.14228

baseline_bills = (
    pilot_data.groupby("LCLid")
    .agg(
        intervals=("energy_kwh", "size"),
        annual_kwh=("energy_kwh", "sum"),
    )
    .reset_index()
)

baseline_bills["flat_energy_charge_gbp"] = (
    baseline_bills["annual_kwh"] * flat_rate_gbp_per_kwh
)

print("Baseline rate (GBP/kWh):", flat_rate_gbp_per_kwh)
print("Energy charges only; standing charges are excluded.")

display(
    baseline_bills.round({
        "annual_kwh": 3,
        "flat_energy_charge_gbp": 2,
    })
)

baseline_file = processed_folder / "pilot_flat_rate_baseline_2013.csv"
baseline_bills.to_csv(baseline_file, index=False)

print("Baseline saved:", baseline_file)

Baseline rate (GBP/kWh): 0.14228
Energy charges only; standing charges are excluded.


,LCLid,intervals,annual_kwh,flat_energy_charge_gbp
0,MAC000002,17520,4214.925,599.7
1,MAC000006,17520,1053.865,149.94
2,MAC000010,17520,9309.05,1324.49
3,MAC000018,17520,3723.334,529.76
4,MAC000019,17520,2309.475,328.59
5,MAC000027,17520,3174.722,451.7


Baseline saved: D:\electricity-tariff-optimisation\data\processed\pilot_flat_rate_baseline_2013.csv


In [17]:
local_start = (
    pilot_data["interval_start_utc"]
    .dt.tz_convert("Europe/London")
)

hour = local_start.dt.hour
weekday = local_start.dt.dayofweek.lt(5)

pilot_data["candidate_band"] = "Shoulder"

pilot_data.loc[
    hour.lt(7),
    "candidate_band",
] = "Off-peak"

pilot_data.loc[
    weekday & hour.ge(16) & hour.lt(20),
    "candidate_band",
] = "Peak"

usage_by_band = (
    pilot_data.groupby(["LCLid", "candidate_band"])["energy_kwh"]
    .sum()
    .unstack(fill_value=0)
    .reindex(columns=["Peak", "Shoulder", "Off-peak"], fill_value=0)
)

usage_share = (
    usage_by_band.div(usage_by_band.sum(axis=1), axis=0) * 100
)

assert (
    usage_by_band.sum(axis=1)
    .sub(baseline_bills.set_index("LCLid")["annual_kwh"])
    .abs()
    .lt(0.000001)
    .all()
)

print("Candidate bands use UK local time.")
print("Weekdays mean Monday-Friday; bank holidays are not excluded.")

print("\nEnergy by band (kWh):")
display(usage_by_band.round(3))

print("\nEnergy share by band (%):")
display(usage_share.round(2))

Candidate bands use UK local time.
Weekdays mean Monday-Friday; bank holidays are not excluded.

Energy by band (kWh):


candidate_band,Peak,Shoulder,Off-peak
LCLid,,,
MAC000002,664.571,2781.715,768.639
MAC000006,166.605,719.809,167.451
MAC000010,1739.576,6375.87,1193.604
MAC000018,474.154,2217.703,1031.477
MAC000019,409.075,1542.475,357.925
MAC000027,309.272,2110.57,754.88



Energy share by band (%):


candidate_band,Peak,Shoulder,Off-peak
LCLid,,,
MAC000002,15.77,66.0,18.24
MAC000006,15.81,68.3,15.89
MAC000010,18.69,68.49,12.82
MAC000018,12.73,59.56,27.7
MAC000019,17.71,66.79,15.5
MAC000027,9.74,66.48,23.78


In [18]:
candidate_rates = pd.Series({
    "Peak": 0.20,
    "Shoulder": 0.14,
    "Off-peak": 0.08,
})

comparison = baseline_bills.set_index("LCLid").copy()

comparison["candidate_energy_charge_gbp"] = (
    usage_by_band.mul(candidate_rates, axis="columns").sum(axis=1)
)

comparison["saving_gbp"] = (
    comparison["flat_energy_charge_gbp"]
    - comparison["candidate_energy_charge_gbp"]
)

comparison["saving_pct"] = (
    comparison["saving_gbp"]
    / comparison["flat_energy_charge_gbp"]
    * 100
)

print("Illustrative tariff scenario; rates are not optimised.")
print("Positive savings mean lower energy charges.")
print("Standing charges and changes in consumption are excluded.")

display(
    comparison[[
        "annual_kwh",
        "flat_energy_charge_gbp",
        "candidate_energy_charge_gbp",
        "saving_gbp",
        "saving_pct",
    ]].round(2)
)

comparison.to_csv(
    processed_folder / "pilot_candidate_tariff_comparison_2013.csv"
)

Illustrative tariff scenario; rates are not optimised.
Positive savings mean lower energy charges.
Standing charges and changes in consumption are excluded.


,annual_kwh,flat_energy_charge_gbp,candidate_energy_charge_gbp,saving_gbp,saving_pct
LCLid,,,,,
MAC000002,4214.93,599.7,583.85,15.85,2.64
MAC000006,1053.86,149.94,147.49,2.45,1.64
MAC000010,9309.05,1324.49,1336.03,-11.53,-0.87
MAC000018,3723.33,529.76,487.83,41.93,7.91
MAC000019,2309.47,328.59,326.4,2.2,0.67
MAC000027,3174.72,451.7,417.72,33.97,7.52


In [19]:
interval_rates = pilot_data["candidate_band"].map(candidate_rates)

assert interval_rates.notna().all()

interval_charges = (
    pilot_data["energy_kwh"] * interval_rates
)

direct_bills = interval_charges.groupby(
    pilot_data["LCLid"]
).sum()

bill_difference = (
    direct_bills
    - comparison["candidate_energy_charge_gbp"]
).abs()

assert bill_difference.lt(0.000001).all()

summary = pd.Series({
    "households": len(comparison),
    "households_paying_less": int(
        comparison["saving_gbp"].gt(0).sum()
    ),
    "households_paying_more": int(
        comparison["saving_gbp"].lt(0).sum()
    ),
    "total_baseline_charge_gbp": (
        comparison["flat_energy_charge_gbp"].sum()
    ),
    "total_candidate_charge_gbp": (
        comparison["candidate_energy_charge_gbp"].sum()
    ),
    "total_saving_gbp": comparison["saving_gbp"].sum(),
})

print("Interval-level vs aggregated billing: PASS")
display(summary.round(2))

summary.to_csv(
    quality_folder / "pilot_tariff_calculation_summary.csv",
    header=["value"],
)

Interval-level vs aggregated billing: PASS


households                       6.00
households_paying_less           5.00
households_paying_more           1.00
total_baseline_charge_gbp     3384.18
total_candidate_charge_gbp    3299.31
total_saving_gbp                84.87
dtype: float64

In [20]:
scenario_costs = pd.Series({
    "Peak": 0.15,
    "Shoulder": 0.10,
    "Off-peak": 0.06,
})

comparison["scenario_variable_cost_gbp"] = (
    usage_by_band.mul(
        scenario_costs,
        axis="columns",
    ).sum(axis=1)
)

comparison["baseline_contribution_gbp"] = (
    comparison["flat_energy_charge_gbp"]
    - comparison["scenario_variable_cost_gbp"]
)

comparison["candidate_contribution_gbp"] = (
    comparison["candidate_energy_charge_gbp"]
    - comparison["scenario_variable_cost_gbp"]
)

comparison["candidate_contribution_margin_pct"] = (
    comparison["candidate_contribution_gbp"]
    / comparison["candidate_energy_charge_gbp"]
    * 100
)

print("Illustrative variable-cost scenario only.")
print("Contribution = energy revenue minus assumed variable costs.")
print("This is not net profit; fixed costs and other charges are excluded.")

display(
    comparison[[
        "scenario_variable_cost_gbp",
        "baseline_contribution_gbp",
        "candidate_contribution_gbp",
        "candidate_contribution_margin_pct",
    ]].round(2)
)

portfolio_revenue = comparison["candidate_energy_charge_gbp"].sum()
portfolio_contribution = comparison["candidate_contribution_gbp"].sum()

print(
    "Portfolio contribution margin (%):",
    round(portfolio_contribution / portfolio_revenue * 100, 2),
)

Illustrative variable-cost scenario only.
Contribution = energy revenue minus assumed variable costs.
This is not net profit; fixed costs and other charges are excluded.


,scenario_variable_cost_gbp,baseline_contribution_gbp,candidate_contribution_gbp,candidate_contribution_margin_pct
LCLid,,,,
MAC000002,423.98,175.72,159.87,27.38
MAC000006,107.02,42.93,40.47,27.44
MAC000010,970.14,354.35,365.89,27.39
MAC000018,354.78,174.97,133.05,27.27
MAC000019,237.08,91.51,89.31,27.36
MAC000027,302.74,148.96,114.98,27.53


Portfolio contribution margin (%): 27.39


In [21]:
from itertools import product
import numpy as np

bands = ["Peak", "Shoulder", "Off-peak"]
energy_matrix = usage_by_band[bands]

baseline = comparison.loc[
    energy_matrix.index,
    "flat_energy_charge_gbp",
]

portfolio_cost = comparison.loc[
    energy_matrix.index,
    "scenario_variable_cost_gbp",
].sum()

results = []

for peak, shoulder, off_peak in product(
    [0.18, 0.20, 0.22, 0.24],
    [0.12, 0.13, 0.14, 0.15],
    [0.06, 0.08, 0.10],
):
    if not peak > shoulder > off_peak:
        continue

    bills = energy_matrix.mul(
        [peak, shoulder, off_peak],
        axis="columns",
    ).sum(axis=1)

    revenue = bills.sum()
    margin = (revenue - portfolio_cost) / revenue
    increases = bills / baseline - 1

    results.append({
        "peak_rate": peak,
        "shoulder_rate": shoulder,
        "off_peak_rate": off_peak,
        "revenue_gbp": revenue,
        "contribution_margin_pct": margin * 100,
        "total_customer_saving_gbp": baseline.sum() - revenue,
        "max_bill_increase_pct": increases.max() * 100,
        "feasible": (
            margin >= 0.25 - 1e-9
            and increases.max() <= 0.02 + 1e-9
        ),
    })

search_results = pd.DataFrame(results)

feasible_results = (
    search_results.loc[search_results["feasible"]]
    .sort_values("revenue_gbp")
    .reset_index(drop=True)
)

print("Price combinations evaluated:", len(search_results))
print("Feasible combinations:", len(feasible_results))
print("Rates are GBP/kWh; standing charges are excluded.")

display(feasible_results.head(5).round(4))

search_results.to_csv(
    processed_folder / "pilot_tariff_grid_search_2013.csv",
    index=False,
)

Price combinations evaluated: 48
Feasible combinations: 12
Rates are GBP/kWh; standing charges are excluded.


,peak_rate,shoulder_rate,off_peak_rate,revenue_gbp,contribution_margin_pct,total_customer_saving_gbp,max_bill_increase_pct,feasible
0,0.24,0.13,0.06,3206.8777,25.2937,177.3048,-0.4918,True
1,0.20,0.14,0.06,3213.8290,25.4553,170.3535,-0.9316,True
2,0.22,0.13,0.08,3217.0922,25.5309,167.0904,-1.3162,True
3,0.24,0.12,0.10,3220.3554,25.6063,163.8272,-1.7009,True
4,0.18,0.14,0.08,3224.0435,25.6914,160.1391,-1.7560,True


In [22]:
best_plan = feasible_results.iloc[0]

best_rates = pd.Series({
    "Peak": best_plan["peak_rate"],
    "Shoulder": best_plan["shoulder_rate"],
    "Off-peak": best_plan["off_peak_rate"],
})

best_customer_impact = baseline_bills.set_index("LCLid").copy()

best_customer_impact["candidate_charge_gbp"] = (
    usage_by_band.mul(best_rates, axis="columns").sum(axis=1)
)

best_customer_impact["saving_gbp"] = (
    best_customer_impact["flat_energy_charge_gbp"]
    - best_customer_impact["candidate_charge_gbp"]
)

best_customer_impact["saving_pct"] = (
    best_customer_impact["saving_gbp"]
    / best_customer_impact["flat_energy_charge_gbp"]
    * 100
)

display(
    best_customer_impact[[
        "annual_kwh",
        "flat_energy_charge_gbp",
        "candidate_charge_gbp",
        "saving_gbp",
        "saving_pct",
    ]].sort_values("saving_pct").round(2)
)

best_customer_impact.to_csv(
    processed_folder / "pilot_selected_tariff_customer_impact_2013.csv"
)

,annual_kwh,flat_energy_charge_gbp,candidate_charge_gbp,saving_gbp,saving_pct
LCLid,,,,,
MAC000010,9309.05,1324.49,1317.98,6.51,0.49
MAC000019,2309.47,328.59,320.18,8.42,2.56
MAC000006,1053.86,149.94,143.61,6.34,4.23
MAC000002,4214.93,599.7,567.24,32.46,5.41
MAC000018,3723.33,529.76,463.99,65.77,12.41
MAC000027,3174.72,451.7,393.89,57.81,12.8


In [23]:
best_plan = feasible_results.iloc[0]

best_rates = pd.Series({
    "Peak": best_plan["peak_rate"],
    "Shoulder": best_plan["shoulder_rate"],
    "Off-peak": best_plan["off_peak_rate"],
})

best_customer_impact = baseline_bills.set_index("LCLid").copy()

best_customer_impact["candidate_charge_gbp"] = (
    usage_by_band.mul(best_rates, axis="columns").sum(axis=1)
)

best_customer_impact["saving_gbp"] = (
    best_customer_impact["flat_energy_charge_gbp"]
    - best_customer_impact["candidate_charge_gbp"]
)

best_customer_impact["saving_pct"] = (
    best_customer_impact["saving_gbp"]
    / best_customer_impact["flat_energy_charge_gbp"]
    * 100
)

display(
    best_customer_impact[[
        "annual_kwh",
        "flat_energy_charge_gbp",
        "candidate_charge_gbp",
        "saving_gbp",
        "saving_pct",
    ]].sort_values("saving_pct").round(2)
)

best_customer_impact.to_csv(
    processed_folder / "pilot_selected_tariff_customer_impact_2013.csv"
)

,annual_kwh,flat_energy_charge_gbp,candidate_charge_gbp,saving_gbp,saving_pct
LCLid,,,,,
MAC000010,9309.05,1324.49,1317.98,6.51,0.49
MAC000019,2309.47,328.59,320.18,8.42,2.56
MAC000006,1053.86,149.94,143.61,6.34,4.23
MAC000002,4214.93,599.7,567.24,32.46,5.41
MAC000018,3723.33,529.76,463.99,65.77,12.41
MAC000027,3174.72,451.7,393.89,57.81,12.8


In [24]:
split_time = pd.Timestamp("2013-07-01", tz="UTC")

pricing_data = pilot_data.copy()

pricing_data["period"] = "Design"
pricing_data.loc[
    pricing_data["interval_start_utc"].ge(split_time),
    "period",
] = "Validation"

period_usage = (
    pricing_data.groupby(
        ["period", "LCLid", "candidate_band"]
    )["energy_kwh"]
    .sum()
    .unstack(fill_value=0)
    .reindex(
        columns=["Peak", "Shoulder", "Off-peak"],
        fill_value=0,
    )
)

period_summary = pricing_data.groupby("period").agg(
    households=("LCLid", "nunique"),
    rows=("energy_kwh", "size"),
    total_kwh=("energy_kwh", "sum"),
    start=("interval_start_utc", "min"),
    last_interval_start=("interval_start_utc", "max"),
)

assert len(pricing_data) == len(pilot_data)

assert np.isclose(
    period_usage.to_numpy().sum(),
    pilot_data["energy_kwh"].sum(),
)

display(period_summary)
display(period_usage.round(3))

,households,rows,total_kwh,start,last_interval_start
period,,,,,
Design,6,52128,12180.671998,2013-01-01 00:00:00+00:00,2013-06-30 23:30:00+00:00
Validation,6,52992,11604.699,2013-07-01 00:00:00+00:00,2013-12-31 23:30:00+00:00


candidate_band           Peak  Shoulder  Off-peak
period     LCLid                                 
Design     MAC000002  331.363  1470.438   385.727
           MAC000006   86.039   376.441    83.193
           MAC000010   845.93  3186.804   601.666
           MAC000018  249.475  1146.133   517.142
           MAC000019  218.542   804.097   168.505
           MAC000027  167.044  1127.319   414.814
Validation MAC000002  333.208  1311.277   382.912
           MAC000006   80.566   343.368    84.258
           MAC000010  893.646  3189.066   591.938
           MAC000018  224.679   1071.57   514.335
           MAC000019  190.533   738.378    189.42
           MAC000027  142.228   983.251   340.066

In [25]:
bands = ["Peak", "Shoulder", "Off-peak"]

design_usage = period_usage.loc["Design", bands]
validation_usage = period_usage.loc["Validation", bands]

design_baseline = (
    design_usage.sum(axis=1) * flat_rate_gbp_per_kwh
)

design_cost = design_usage.mul(
    scenario_costs[bands],
    axis="columns",
).sum(axis=1)

design_results = []

for peak, shoulder, off_peak in product(
    [0.18, 0.20, 0.22, 0.24],
    [0.12, 0.13, 0.14, 0.15],
    [0.06, 0.08, 0.10],
):
    if not peak > shoulder > off_peak:
        continue

    rates = pd.Series(
        [peak, shoulder, off_peak],
        index=bands,
    )

    bills = design_usage.mul(rates, axis="columns").sum(axis=1)
    revenue = bills.sum()
    margin = (revenue - design_cost.sum()) / revenue
    max_increase = (bills / design_baseline - 1).max()

    design_results.append({
        "peak_rate": peak,
        "shoulder_rate": shoulder,
        "off_peak_rate": off_peak,
        "revenue_gbp": revenue,
        "margin_pct": margin * 100,
        "max_bill_increase_pct": max_increase * 100,
        "feasible": (
            margin >= 0.25 - 1e-9
            and max_increase <= 0.02 + 1e-9
        ),
    })

design_search = pd.DataFrame(design_results)

design_feasible = (
    design_search.loc[design_search["feasible"]]
    .sort_values(
        ["revenue_gbp", "peak_rate", "shoulder_rate", "off_peak_rate"]
    )
    .reset_index(drop=True)
)

if design_feasible.empty:
    raise ValueError("No feasible tariff in the design period.")

selected = design_feasible.iloc[0]

locked_rates = pd.Series({
    "Peak": selected["peak_rate"],
    "Shoulder": selected["shoulder_rate"],
    "Off-peak": selected["off_peak_rate"],
})

evaluation_rows = []

for period in ["Design", "Validation"]:
    usage = period_usage.loc[period, bands]

    baseline_charge = usage.sum(axis=1) * flat_rate_gbp_per_kwh
    candidate_charge = usage.mul(
        locked_rates, axis="columns"
    ).sum(axis=1)
    cost = usage.mul(
        scenario_costs[bands], axis="columns"
    ).sum(axis=1)

    revenue = candidate_charge.sum()
    margin = (revenue - cost.sum()) / revenue
    increases = candidate_charge / baseline_charge - 1

    evaluation_rows.append({
        "period": period,
        "baseline_charge_gbp": baseline_charge.sum(),
        "candidate_charge_gbp": revenue,
        "customer_saving_gbp": (
            baseline_charge.sum() - revenue
        ),
        "contribution_margin_pct": margin * 100,
        "max_bill_increase_pct": increases.max() * 100,
        "households_paying_more": int(increases.gt(1e-9).sum()),
        "margin_constraint_pass": margin >= 0.25 - 1e-9,
        "bill_constraint_pass": increases.max() <= 0.02 + 1e-9,
    })

period_evaluation = pd.DataFrame(evaluation_rows)

print("Rates selected using Design data only (GBP/kWh):")
display(locked_rates.to_frame("rate"))

print("Same rates applied to both periods:")
display(period_evaluation.round(3))

design_search.to_csv(
    processed_folder / "pilot_design_tariff_search.csv",
    index=False,
)

period_evaluation.to_csv(
    processed_folder / "pilot_locked_tariff_period_evaluation.csv",
    index=False,
)

Rates selected using Design data only (GBP/kWh):


,rate
Peak,0.24
Shoulder,0.13
Off-peak,0.06


Same rates applied to both periods:


,period,baseline_charge_gbp,candidate_charge_gbp,customer_saving_gbp,contribution_margin_pct,max_bill_increase_pct,households_paying_more,margin_constraint_pass,bill_constraint_pass
0,Design,1733.066,1640.337,92.729,25.250,-0.906,0,True,True
1,Validation,1651.117,1566.540,84.576,25.339,-0.081,0,True,True


In [26]:
validation_revenue = validation_usage.mul(
    locked_rates,
    axis="columns",
).to_numpy().sum()

base_variable_cost = validation_usage.mul(
    scenario_costs[bands],
    axis="columns",
).to_numpy().sum()

stress_rows = []

for increase_pct in [0, 5, 10, 20]:
    stressed_cost = base_variable_cost * (1 + increase_pct / 100)
    contribution = validation_revenue - stressed_cost
    margin_pct = contribution / validation_revenue * 100

    stress_rows.append({
        "cost_increase_pct": increase_pct,
        "revenue_gbp": validation_revenue,
        "variable_cost_gbp": stressed_cost,
        "contribution_gbp": contribution,
        "contribution_margin_pct": margin_pct,
        "margin_constraint_pass": margin_pct >= 25 - 1e-7,
    })

cost_stress = pd.DataFrame(stress_rows)

display(cost_stress.round(3))

cost_stress.to_csv(
    processed_folder / "pilot_validation_cost_stress.csv",
    index=False,
)

,cost_increase_pct,revenue_gbp,variable_cost_gbp,contribution_gbp,contribution_margin_pct,margin_constraint_pass
0,0,1566.54,1169.596,396.945,25.339,True
1,5,1566.54,1228.076,338.465,21.606,False
2,10,1566.54,1286.555,279.985,17.873,False
3,20,1566.54,1403.515,163.026,10.407,False


In [27]:
import json

pilot_config = {
    "status": "development_pilot",
    "household_ids": list(complete_ids),
    "currency": "GBP",
    "baseline_rate_gbp_per_kwh": flat_rate_gbp_per_kwh,
    "selected_rates_gbp_per_kwh": locked_rates.to_dict(),
    "assumed_variable_costs_gbp_per_kwh": scenario_costs.to_dict(),
    "constraints": {
        "minimum_portfolio_contribution_margin_pct": 25,
        "maximum_customer_bill_increase_pct": 2,
    },
    "candidate_bands": {
        "timezone": "Europe/London",
        "peak": "Monday-Friday, 16:00-20:00",
        "off_peak": "Every day, 00:00-07:00",
        "shoulder": "All remaining intervals",
        "bank_holidays_excluded": False,
    },
    "periods": {
        "basis": "UTC interval start",
        "design": "2013-01-01 to 2013-06-30",
        "validation": "2013-07-01 to 2013-12-31",
    },
    "limitations": [
        "Six households selected from the first source file.",
        "Annual results were inspected before time-split evaluation.",
        "Prices selected from a finite candidate grid.",
        "Costs are illustrative assumptions, not observed supplier costs.",
        "Consumption is held constant across tariff scenarios.",
        "Standing charges and fixed costs are excluded.",
        "Original dynamic tariff schedule is not used in this scenario.",
    ],
}

config_file = processed_folder / "pilot_tariff_config.json"

config_file.write_text(
    json.dumps(pilot_config, indent=2),
    encoding="utf-8",
)

print("Pilot configuration saved:", config_file)

Pilot configuration saved: D:\electricity-tariff-optimisation\data\processed\pilot_tariff_config.json
